# Capstone Project: Loan Default Prediction System
### End-to-End Machine Learning Pipeline for Credit Risk Assessment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![Dataset](https://img.shields.io/badge/Dataset-Kaggle%2FBank_Loan-blue)](https://www.kaggle.com/datasets/itsmesunil/bank-loan-modelling)
[![Python](https://img.shields.io/badge/Python-3.10%2B-brightgreen)](https://www.python.org/)
[![Scikit-Learn](https://img.shields.io/badge/Library-Scikit--Learn-orange)](https://scikit-learn.org/)

---

### Project Overview
In retail banking, assessing credit risk is essential for minimizing default loss while approving creditworthy applicants. The goal of this project is to develop an automated classification pipeline to predict whether an applicant will repay (`Y`) or default (`N`) on their loan.

#### Workflow:
1. Environment Setup & Data Ingestion
2. Data Cleaning & Missing Value Imputation
3. Feature Engineering
4. Exploratory Data Analysis (EDA)
5. Model Benchmarking (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting)
6. Model Evaluation & Diagnostics (Confusion Matrix, ROC-AUC)
7. Hyperparameter Tuning (GridSearchCV)
8. Feature Importance & Business Interpretations

In [ ]:
import os
import sys
import urllib.request
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

# Download dataset if not found locally (supports Google Colab and Kaggle runtimes)
data_dir = "data"
data_path = os.path.join(data_dir, "loan_data.csv")

if not os.path.exists(data_path) and not os.path.exists("loan_data.csv"):
    os.makedirs(data_dir, exist_ok=True)
    raw_url = "https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv"
    urllib.request.urlretrieve(raw_url, data_path)
    print("Dataset downloaded to:", data_path)
else:
    if os.path.exists("loan_data.csv"):
        data_path = "loan_data.csv"
    print("Using dataset at:", data_path)

## 1. Data Ingestion & Initial Inspection

In [ ]:
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("-" * 75)
print(df.info())
print("-" * 75)
display(df.head(5))

### 1.1 Summary Statistics & Missing Values Analysis

In [ ]:
display(df.describe().T)

null_summary = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
null_summary = null_summary[null_summary['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)
display(null_summary)

## 2. Data Cleaning & Imputation
- Categorical variables are imputed with the mode.
- Numerical variables are imputed with the median to resist outlier skew.

In [ ]:
df_clean = df.copy()

# Mode imputation for categorical attributes
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History']:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

# Median imputation for numerical attributes
for col in ['LoanAmount', 'Loan_Amount_Term']:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

assert df_clean.isnull().sum().sum() == 0, "Missing values remain after imputation"
print("Missing values successfully imputed. Remaining nulls: 0")

## 3. Feature Engineering
- `TotalIncome`: Household income combining applicant and co-applicant earnings.
- `EMI`: Estimated monthly installment approximation.
- `BalanceIncome`: Residual disposable income after paying the monthly loan installment.
- `Log Transformations`: Applied to skewed distributions (`LoanAmount`, `TotalIncome`).

In [ ]:
# Total household income
df_clean['TotalIncome'] = df_clean['ApplicantIncome'] + df_clean['CoapplicantIncome']

# Monthly EMI approximation (LoanAmount is in thousands USD, Term is in months)
df_clean['EMI'] = (df_clean['LoanAmount'] * 1000) / df_clean['Loan_Amount_Term']

# Remaining monthly balance after installment
df_clean['BalanceIncome'] = df_clean['TotalIncome'] - df_clean['EMI']

# Log transformations for skewed continuous features
df_clean['LoanAmount_Log'] = np.log1p(df_clean['LoanAmount'])
df_clean['TotalIncome_Log'] = np.log1p(df_clean['TotalIncome'])
df_clean['ApplicantIncome_Log'] = np.log1p(df_clean['ApplicantIncome'])

display(df_clean[['ApplicantIncome', 'CoapplicantIncome', 'TotalIncome', 'LoanAmount', 'EMI', 'BalanceIncome', 'LoanAmount_Log', 'TotalIncome_Log']].head(4))

## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Donut chart
approval_counts = df_clean['Loan_Status'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(approval_counts, labels=['Approved (Y)', 'Default/Rejected (N)'], autopct='%1.1f%%',
            startangle=140, colors=colors, explode=(0.05, 0), wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].add_artist(plt.Circle((0,0), 0.60, fc='white'))
axes[0].set_title("Target Distribution: Loan Status", fontsize=13, fontweight='bold')

# Countplot
sns.countplot(x='Loan_Status', data=df_clean, palette=colors, ax=axes[1], order=['Y', 'N'])
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height())}", (p.get_x() + 0.35, p.get_height() / 2),
                     color='white', fontweight='bold', fontsize=12)
axes[1].set_title("Approval vs. Default Counts", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Loan Status")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### 4.1 Categorical Predictors vs. Loan Status

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
features = ['Credit_History', 'Property_Area', 'Education', 'Married']
titles = [
    'Credit History vs Approval',
    'Property Area vs Approval',
    'Education vs Approval',
    'Marital Status vs Approval'
]

for idx, (col, title) in enumerate(zip(features, titles)):
    ax = axes[idx // 2, idx % 2]
    cross_tab = pd.crosstab(df_clean[col], df_clean['Loan_Status'], normalize='index') * 100
    cross_tab[['Y', 'N']].plot(kind='bar', stacked=True, color=['#2ecc71', '#e74c3c'], ax=ax, alpha=0.9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel(col)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(['Approved (Y)', 'Rejected (N)'], loc='upper right')

plt.tight_layout()
plt.show()

### 4.2 Skewness Reduction via Log Transformation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Before log transform
sns.histplot(df_clean['TotalIncome'], kde=True, color='#3498db', ax=axes[0, 0], bins=30)
axes[0, 0].set_title(f"Total Income Distribution (Skew: {df_clean['TotalIncome'].skew():.2f})", fontsize=12)

sns.histplot(df_clean['LoanAmount'], kde=True, color='#e67e22', ax=axes[0, 1], bins=30)
axes[0, 1].set_title(f"Loan Amount Distribution (Skew: {df_clean['LoanAmount'].skew():.2f})", fontsize=12)

# After log transform
sns.histplot(df_clean['TotalIncome_Log'], kde=True, color='#2ecc71', ax=axes[1, 0], bins=30)
axes[1, 0].set_title(f"Log(Total Income) Distribution (Skew: {df_clean['TotalIncome_Log'].skew():.2f})", fontsize=12)

sns.histplot(df_clean['LoanAmount_Log'], kde=True, color='#9b59b6', ax=axes[1, 1], bins=30)
axes[1, 1].set_title(f"Log(Loan Amount) Distribution (Skew: {df_clean['LoanAmount_Log'].skew():.2f})", fontsize=12)

plt.tight_layout()
plt.show()

### 4.3 Correlation Analysis

In [ ]:
corr_df = df_clean.copy()
corr_df['Loan_Status_Numeric'] = corr_df['Loan_Status'].map({'Y': 1, 'N': 0})

num_cols_for_corr = [
    'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term',
    'Credit_History', 'TotalIncome', 'EMI', 'BalanceIncome',
    'LoanAmount_Log', 'TotalIncome_Log', 'Loan_Status_Numeric'
]

plt.figure(figsize=(11, 8))
corr_matrix = corr_df[num_cols_for_corr].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title("Correlation Matrix of Financial Attributes & Target", fontsize=13, fontweight='bold')
plt.show()

## 5. Preprocessing & Data Preparation
- Remove `Loan_ID` and raw non-transformed columns.
- Binary encode target: $Y \to 1, N \to 0$.
- One-hot encode remaining categorical variables.
- Split data with an 80:20 stratified holdout split.
- Standardize features using `StandardScaler` fitted on training data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_model = df_clean.copy()
cols_to_drop = ['Loan_ID', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'TotalIncome']
data_model.drop(columns=cols_to_drop, inplace=True)

data_model['Loan_Status'] = data_model['Loan_Status'].map({'Y': 1, 'N': 0})

categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
data_model = pd.get_dummies(data_model, columns=categorical_cols, drop_first=True)

X = data_model.drop(columns=['Loan_Status'])
y = data_model['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

feature_names = X.columns.tolist()
print(f"Training samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]} | Features: {len(feature_names)}")

## 6. Supervised Model Benchmarking
Evaluating four classification algorithms:
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. Gradient Boosting Classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=4),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3)
}

benchmark_results = []
trained_models = {}
predictions = {}
prob_predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    predictions[name] = y_pred
    prob_predictions[name] = y_prob
    
    benchmark_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(benchmark_results).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
display(results_df.style.format({
    'Accuracy': '{:.3f}',
    'Precision': '{:.3f}',
    'Recall': '{:.3f}',
    'F1-Score': '{:.3f}',
    'ROC-AUC': '{:.3f}'
}))

## 7. Model Diagnostic Evaluation
### 7.1 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

for idx, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[idx // 2, idx % 2]
    cm = confusion_matrix(y_test, y_pred)
    cm_labels = [f"{count}\n({pct:.1%})" for count, pct in zip(cm.flatten(), cm.flatten() / np.sum(cm))]
    cm_labels = np.asarray(cm_labels).reshape(2, 2)
    
    sns.heatmap(cm, annot=cm_labels, fmt='', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Pred: Default (0)', 'Pred: Repay (1)'],
                yticklabels=['Actual: Default (0)', 'Actual: Repay (1)'])
    ax.set_title(f"{name}\nAccuracy: {accuracy_score(y_test, y_pred):.2%}", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 7.2 Comparative ROC-AUC Curves

In [ ]:
plt.figure(figsize=(10, 7))
colors_dict = {
    'Logistic Regression': '#2980b9',
    'Decision Tree': '#e67e22',
    'Random Forest': '#27ae60',
    'Gradient Boosting': '#8e44ad'
}

for name, y_prob in prob_predictions.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.3f})", color=colors_dict[name], linewidth=2.5)

plt.plot([0, 1], [0, 1], 'k--', label="Chance (AUC = 0.500)", alpha=0.6)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity / Recall)")
plt.title("Multi-Model ROC-AUC Curve Comparison", fontsize=13, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Hyperparameter Optimization (GridSearchCV)
Optimizing the Random Forest classifier using 5-fold cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 8, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)

grid_rf.fit(X_train_scaled, y_train)
best_rf = grid_rf.best_estimator_

print("Optimal parameters:", grid_rf.best_params_)

tuned_pred = best_rf.predict(X_test_scaled)
tuned_prob = best_rf.predict_proba(X_test_scaled)[:, 1]

print(f"Tuned Random Forest Test Accuracy: {accuracy_score(y_test, tuned_pred):.3%}")
print(f"Tuned Random Forest F1-Score:      {f1_score(y_test, tuned_pred):.3f}")
print(f"Tuned Random Forest ROC-AUC:       {roc_auc_score(y_test, tuned_prob):.3f}")

## 9. Feature Importance Analysis
Assessing predictor impact via Gini importance from the tuned Random Forest model.

In [ ]:
importances = best_rf.feature_importances_
feat_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(12, 7))
bars = plt.barh(feat_importance_df['Feature'], feat_importance_df['Importance'], color='#3498db')
bars[-1].set_color('#e74c3c')

plt.title("Gini Feature Importance (Tuned Random Forest)", fontsize=13, fontweight='bold')
plt.xlabel("Importance Score")
plt.ylabel("Feature")

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.005, bar.get_y() + bar.get_height()/2, f"{width:.3f}",
             va='center', fontsize=10, color='#2c3e50')

plt.xlim(0, max(importances) * 1.15)
plt.tight_layout()
plt.show()

## 10. Business Findings & Risk Policy Recommendations

### Key Findings:
1. **Credit History**: Strongest predictor. Applicants with compliant credit history (`Credit_History == 1.0`) achieve ~80% approval, while those with delinquencies (`0.0`) drop to under 10%.
2. **Household Income (`TotalIncome`)**: Combined income provides a clearer assessment of debt service capacity than individual applicant income.
3. **Property Area**: Applications in Semiurban areas demonstrate higher repayment track records compared to Rural areas.
4. **Loan-to-Term Balance**: High loan values relative to term increase default risk unless supported by adequate disposable cashflow (`BalanceIncome`).

### Operational Guidelines:
- **Fast-Track Tier**: Applicants with clean credit history and low debt-to-income ratios can be automated for rapid approval.
- **Tenure Adjustment**: Borderline applicants can be offered extended tenures to lower their monthly EMI obligation.
- **High-Risk Threshold**: Applicants without credit history compliance require co-signers or collateral verification before manual review.